## Section 1: Initial Setup

In [2]:
import numpy as np
import os
import pandas as pd
# import fnv
# import fnv.reduce
import matplotlib.pyplot as plt
import cv2
import matplotlib.animation as animation
from skimage.feature import canny
import math
# %matplotlib notebo

# import gui
# import registration_gui as rgui
import SimpleITK as sitk
from ipywidgets import interact,fixed
import itertools
from skimage.transform import hough_circle, hough_circle_peaks
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

import itk
import re

In [3]:
def preprocess_img(img,lower_th_value=85,thresh_type=True):
    """
    img: Normalized Image
    lower_th_value: Value of Below and above which pixel values will be changes

    Process Image frame and gives remove noizy objects and give clear representation
    Reflective Markers on the Hand 

    """
    if thresh_type:
        thesh = cv2.threshold(img,lower_th_value,200,cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)[1]
    else:
        thesh = cv2.threshold(img,lower_th_value,200,cv2.THRESH_BINARY_INV)[1]
   
    # thesh = cv2.threshold(img,lower_th_value,130,cv2.THRESH_BINARY_INV)[1]
    edges = canny(thesh,sigma=3,low_threshold=5,high_threshold=20)
    return edges,thesh


In [4]:

def normalize(image):
    """
    Normalize Image Based on the Min & Maximum of the Image
    """
    return (255.0 * (image-np.min(image))/np.ptp(image)).astype(np.uint8)

In [5]:
def visualize_image(image,circle_info):
    """

    Draw Image Circle
    """

    
    for center_x, center_y,radius in circle_info:
        cv2.circle(image,(center_x,center_y),int(radius),(255,255,0),1,-1)

    return image
    

In [6]:

def find_hough_circles(edge_image,first=True):

    # Radius range in pixels
    # Radius range for the markers: This is something you want to manually calculate for each video
    # You can refer to excel sheet: Which has added manual radius of each video.
    hough_radii =np.arange(8.5,12,0.1)

    hough_res = hough_circle(edge_image, hough_radii)


    # Number of Peaks in Circle Detection: This is a hyperparamter(Optional)
    # 
    if first:
        noOfPeaks = 4
        accums, cx, cy, radii = hough_circle_peaks(hough_res, hough_radii,num_peaks=1, total_num_peaks=noOfPeaks,min_xdistance=10)
    else:
        accums, cx, cy, radii = hough_circle_peaks(hough_res, hough_radii,num_peaks=1, total_num_peaks=1,min_xdistance=10)

    circle_info = []
    for center_y, center_x, rx in zip(cy, cx, radii):
        circle_info.append([center_x,center_y,rx])

    return circle_info



In [7]:
def getting_window(image,circle_info):

    height, width = image.shape[:2]
    pad_amt = 15

    windows = []
    updated_centroid= []
    for idx,(center_x, center_y,radius) in enumerate(circle_info):
        

        print(f' Radius:{radius}')

        x = round(center_x)
        y = round(center_y)


        # Padding should not exceeded the image dimensions.
        if (x - pad_amt >= 0 and x + pad_amt < width and y - pad_amt >= 0 and y + pad_amt < height):
            # Extract the 8x8 window around the given pixel
            window = image[y-pad_amt:y+pad_amt, x-pad_amt:x+pad_amt]
        
        # If adding padded resulted in minus corrdinates from X coordinate( Width)
        elif (x-pad_amt <0):

        
            window = image[y-pad_amt:height, x-pad_amt:x+pad_amt]

            print(f'There is some wrong with these x:{x} and y:{y} coordinates')

       
        # As video progress, threshold value of markers can be different, therefore if the algorithm fails
        # Try different Threshold Values and Methods as shown below.
        # if idx == 0:
        #     edge_window,thresh_window = preprocess_img(window,lower_th_value=40,thresh_type=False)
        # else:
        #     edge_window,thresh_window = preprocess_img(window,lower_th_value=45)

        # edge_window,thresh_window = preprocess_img(window,lower_th_value=120,thresh_type=False)
        edge_window,thresh_window = preprocess_img(window)
        


        # Find the hough cicle from given the surface of marker
        circle_info = find_hough_circles(edge_window,first=False)

        
        # Update the center of x,y relative original image frame coordinates.
        new_x_center = (x-pad_amt) + circle_info[0][0]
        new_y_center = (y-pad_amt) + circle_info[0][1]
        
        
        updated_centroid.append([new_x_center,new_y_center,circle_info[0][2]])

        windows.append(window)
    return updated_centroid,windows
       
        
        

    






In [8]:
%matplotlib tk

def get_click_coordinates(image,radius):
    
    coords = []

    fig,ax = plt.subplots()
    ax.imshow(image)
    ax.set_title('Markers Acqusition')

    def onclick(event):

        if event.xdata is not None and event.ydata is not None:
            x,y = event.xdata,event.ydata
            coords.append([x,y,radius])
            ax.plot(x,y,'go')
            fig.canvas.draw()


    cid = fig.canvas.mpl_connect('button_press_event',onclick)
    plt.show()
    return coords

In [9]:
def display_images_with_alpha(image_z,alpha,fixed):

    # print(moving[image_z,:,:])
    # print(fixed)

    img = (1.0-alpha) * fixed[image_z]+ alpha * fixed[image_z]
    plt.imshow(img,cmap='viridis')
    plt.axis('off')
    plt.show()

In [ ]:

# Root Directory of Patient Data: Cropping the video prior to this step is ideal
ROOT = r"/home/boromir/Documents/UM"

# Thermal Referene Image (Prior to any thermal stimulation)
reference_img = "Snap-xxxx.npz"


# Thermal Video (Either cold or Warm Stimulation)
video = "Rec-xxxx.npz"


ref_img = np.load(os.path.join(ROOT,reference_img))['reg_image']


vid = np.load(os.path.join(ROOT,video))['cropped_vid']

# First frame of Thermal Video
first_frame = vid[:,:,0]





## Section 2: Unit Testing


In [12]:

# Get the location of Markers in First Thermal Frame of the Sequence
first_thermalframe_coords = get_click_coordinates(first_frame,radius=8)


In [13]:
# Observe the number of markers that you can see from the first frame of thermal video
n_markers = 3

assert len(first_thermalframe_coords) == n_markers, 'Number of Coordinates must be equal to number of markers'

In [14]:
markers_location = first_thermalframe_coords

In [15]:
frame_idx = 1

move_image = vid[:,:,frame_idx]

# Image after the normalization
mv_norm_image = normalize(move_image)
plt.imshow(mv_norm_image)

In [16]:
new_center,windows = getting_window(mv_norm_image,markers_location)


 Radius:8
 Radius:8
 Radius:8


In [17]:
# idx in range(n_markers)
# View every window which is yielded from markers location of previous frame (0)

idx=0
wind = windows.copy()[idx]
copy_window = windows.copy()[idx]

In [18]:

plt.imshow(wind)

In [19]:
edges,thesh = preprocess_img(wind,lower_th_value=85)

In [20]:
plt.imshow(edges)

In [21]:
plt.imshow(thesh)

In [22]:
hough_radii = np.arange(8.5,12,0.2)
hough_res = hough_circle(edges, hough_radii)
accums, cx, cy, radii = hough_circle_peaks(hough_res, hough_radii,num_peaks=1, total_num_peaks=1,min_xdistance=10)




In [23]:
circle_info = []
for center_y, center_x, radius in zip(cy, cx, radii):
        radius= 9.9
        circle_info.append([center_x,center_y,radius])


In [24]:
%matplotlib tk
plot_image = visualize_image(wind,circle_info)
plt.imshow(plot_image)
plt.show()


In [ ]:
# Can be used to compare how does each image looks like after every pre processing steps
# This is an ideal when teh algorithm fails to identify markers location or identified centers are far away from actual markers, this can be used as a sanity check.

circle_info = find_hough_circles(edges,first=False)

plot_image = visualize_image(wind,circle_info)


fig,ax = plt.subplots(3)

ax[0].imshow(thesh)
ax[1].imshow(edges)
ax[2].imshow(plot_image)




# Manual Image Registration

In [26]:

count = 0
hough_images = []
# 
for idx in range(1,100):

    print(f'Frame is processing {idx}')
    frame_idx = idx
    move_image = vid[:,:,frame_idx]

    
    mv_norm_image = normalize(move_image)

    # plt.imshow(mv_norm_image)
    # plt.show()

    
    new_center,windows = getting_window(mv_norm_image,markers_location)

    plot_image = visualize_image(mv_norm_image,new_center)

    # If algorithm fails to identify the border aroun the marker, then in those frames we have to manully obtain the coordinates
    # if idx == 49:
    #     new_center = [[np.float64(59.84200196270858), np.float64(50.12105705874103), 6],
    #     [np.float64(104.84410486471334), np.float64(43.60206084396464), 6],
    #     [np.float64(122.29819150427595), np.float64(117.62421141174819), 6]]

    # elif idx == 869:
    #    new_center = [[np.float64(60.05229216318524), np.float64(50.962217860647655), 6],
    #     [np.float64(105.05439506519001), np.float64(45.284382447777915), 6],
    #     [np.float64(123.13935230618257), np.float64(118.46537221365482), 6]]

    markers_location = new_center

    # print(circle_info)
    


    hough_images.append(plot_image)
       
   
    
    # break





Frame is processing 1
 Radius:8
 Radius:8
 Radius:8
Frame is processing 2
 Radius:9.199999999999998
 Radius:9.199999999999998
 Radius:8.5
Frame is processing 3
 Radius:9.199999999999998
 Radius:9.199999999999998
 Radius:8.5
Frame is processing 4
 Radius:9.199999999999998
 Radius:9.199999999999998
 Radius:8.5
Frame is processing 5
 Radius:9.199999999999998
 Radius:9.199999999999998
 Radius:8.5
Frame is processing 6
 Radius:9.199999999999998
 Radius:9.199999999999998
 Radius:8.5
Frame is processing 7
 Radius:9.199999999999998
 Radius:9.199999999999998
 Radius:8.5
Frame is processing 8
 Radius:9.199999999999998
 Radius:9.199999999999998
 Radius:8.5
Frame is processing 9
 Radius:9.199999999999998
 Radius:9.199999999999998
 Radius:8.5
Frame is processing 10
 Radius:9.199999999999998
 Radius:9.199999999999998
 Radius:8.5
Frame is processing 11
 Radius:9.199999999999998
 Radius:9.199999999999998
 Radius:8.5
Frame is processing 12
 Radius:9.199999999999998
 Radius:9.199999999999998
 Radius:8.5

In [27]:
def display_images_with_alpha(image_z,alpha,fixed):

    # print(moving[image_z,:,:])
    # print(fixed)

    img = (1.0-alpha) * fixed[image_z]+ alpha * fixed[image_z]
    plt.imshow(img,cmap='viridis')
    plt.axis('off')
    plt.show()

In [ ]:
%matplotlib tk
interact(display_images_with_alpha,
         image_z=(0,len(hough_images)-1),
         alpha = (0,1.0,0.05),
         fixed = fixed(hough_images),
         moving = fixed(hough_images)
         )  
 
  

interactive(children=(IntSlider(value=49, description='image_z', max=98), FloatSlider(value=0.5, description='…

<function __main__.display_images_with_alpha(image_z, alpha, fixed)>

: 

# Full Image Registration Pipeline

In [11]:




# ROOT = r"C:\\Users\\User\\Desktop\\mdh_data\\26_02_2025\\Patient_1\\ThermalData\\Cropped"

# reference_img = "Snap-000097.npz"

# video = "Rec-000098.npz"



# ref_img = np.load(os.path.join(ROOT,reference_img))['reg_image']


vid = np.load(os.path.join(ROOT,video))['cropped_vid']

ref_img = vid[:,:,0]




fixed_image = itk.GetImageFromArray(ref_img)






In [12]:

# Getting the Coordinates of Markers in the Referece Image(Before Thermal Stimulation)
referece_coords =get_click_coordinates(ref_img,9.9)

In [13]:
assert len(referece_coords) == 3, 'Number of Coordinates must be equal to number of markers'

In [14]:
markers_location = referece_coords

In [15]:
referece_coords

[[np.float64(205.32738095238102), np.float64(106.94047619047615), 9.9],
 [np.float64(101.90476190476193), np.float64(81.64285714285708), 9.9],
 [np.float64(91.48809523809524), np.float64(170.18452380952377), 9.9]]

In [16]:
# # Getting the Coordinate of Markers in the First Frame Thermal Sequence(After the thermal Stimulation)

# markers_location = get_click_coordinates(first_frame,radius=7.5)

In [17]:
# assert len(markers_location) == 3, 'Number of Coordinates must be equal to number of markers'

In [18]:
# Remove Radius from each Elements in the Array as it does need to registion
referece_coords= np.array([list(map(float,i[:2])) for i in referece_coords]).tolist()


In [19]:
# update_center

referece_coords

[[205.32738095238102, 106.94047619047615],
 [101.90476190476193, 81.64285714285708],
 [91.48809523809524, 170.18452380952377]]

In [20]:
# This only Required when only two markers as present since Affine transformation does not work with two points
# updated_ref= referece_coords.copy()

In [21]:
# updated_ref

In [22]:
# updated_ref.append([updated_ref[0][0]+updated_ref[0][-1], updated_ref[0][1],updated_ref[0][-1]])

In [23]:
# updated_ref

In [24]:
markers_location

[[np.float64(205.32738095238102), np.float64(106.94047619047615), 9.9],
 [np.float64(101.90476190476193), np.float64(81.64285714285708), 9.9],
 [np.float64(91.48809523809524), np.float64(170.18452380952377), 9.9]]

In [25]:
# updated_ref = np.array([list(map(float,i[:2])) for i in updated_ref]).tolist()

In [28]:
def find_distances(ref_ori,mov_points,no_of_markers=3):
    'Find the Mean Absoulte Differece Between X and Y coodinates'

    return (np.sum(np.abs(np.array(ref_ori) - np.array(mov_points)),axis=0)/no_of_markers).tolist()

In [29]:

count = 0
hough_images = []
manual_count = 0

registered_images = []
transform_matrix = []
LandMarkPoinType = itk.Point[itk.D,2]
LandMarkContainerType = itk.vector[LandMarkPoinType]

fixed_landmarks = LandMarkContainerType()
fixed_point = LandMarkPoinType()
distances = []
overall_distances = []
point_predictions = []

for x in referece_coords:
    fixed_point[0] = x[0]
    fixed_point[1] = x[1]

    fixed_landmarks.push_back(fixed_point)

for idx in range(1,vid.shape[-1]):

    
    frame_idx = idx
    move_image = vid[:,:,frame_idx]

    
    mv_norm_image = normalize(move_image)

    
    
    new_center,windows = getting_window(mv_norm_image,markers_location)

    plot_image = visualize_image(mv_norm_image,new_center)

    
    # if idx ==4625:
    #     markers_location = update_center
    # else:

    #     markers_location = new_center
    markers_location = new_center

    # # Only Requried when 2 markers are present in the Image Sequence
    # manual_coords = markers_location.copy()
    # manual_coords.append([manual_coords[0][0]+manual_coords[0][-1], manual_coords[0][1],manual_coords[0][-1]])   

    mov_points = np.array([list(map(float,i[:2])) for i in markers_location]).tolist()

    point_predictions.append(mov_points)
    moving_image = itk.GetImageFromArray(move_image)


    
    moving_landmarks = LandMarkContainerType()

    moving_point = LandMarkPoinType()


    for x in mov_points:
        moving_point[0] = x[0]
        moving_point [1] = x[1]
        moving_landmarks.push_back(moving_point)
    


    TransformInitializerType = itk.LandmarkBasedTransformInitializer[
    itk.Transform[itk.D,2,2]
]
    
    transform_initializer = TransformInitializerType.New()

    transform_initializer.SetFixedLandmarks(fixed_landmarks)
    transform_initializer.SetMovingLandmarks(moving_landmarks)


    transform = itk.AffineTransform[itk.D,2].New()
    transform_initializer.SetTransform(transform)
    transform_initializer.InitializeTransform()


    output = itk.resample_image_filter(
        moving_image,
        transform=transform,
        use_reference_image=True,
        reference_image = fixed_image,
        default_pixel_value = 0
    )


    moving_inverse = transform.GetInverseTransform()

    pred_coords = [moving_inverse.TransformPoint(i) for i in mov_points]

    pred_coords  = [[pred_coords [i].GetElement(0),pred_coords[i].GetElement(1)]  for i in range(len(pred_coords ))]
    
    overall_distances.append(find_distances(referece_coords,pred_coords))


    final = itk.GetArrayFromImage(output)
    
    registered_images.append(final)
    transform_matrix.append(transform)

    # break
    

 Radius:9.9
 Radius:9.9
 Radius:9.9
 Radius:9.199999999999998
 Radius:9.199999999999998
 Radius:8.5
 Radius:9.199999999999998
 Radius:9.199999999999998
 Radius:8.5
 Radius:9.199999999999998
 Radius:9.199999999999998
 Radius:8.5
 Radius:9.199999999999998
 Radius:9.199999999999998
 Radius:8.5
 Radius:9.199999999999998
 Radius:9.199999999999998
 Radius:8.5
 Radius:9.199999999999998
 Radius:9.199999999999998
 Radius:8.5
 Radius:9.199999999999998
 Radius:9.199999999999998
 Radius:8.5
 Radius:9.199999999999998
 Radius:9.199999999999998
 Radius:8.5
 Radius:9.199999999999998
 Radius:9.199999999999998
 Radius:8.5
 Radius:9.199999999999998
 Radius:9.199999999999998
 Radius:8.5
 Radius:9.199999999999998
 Radius:9.199999999999998
 Radius:8.5
 Radius:9.199999999999998
 Radius:9.199999999999998
 Radius:8.5
 Radius:9.199999999999998
 Radius:9.199999999999998
 Radius:9.199999999999998
 Radius:9.199999999999998
 Radius:9.199999999999998
 Radius:9.199999999999998
 Radius:9.199999999999998
 Radius:9.1999

In [30]:
#  Get Array from Fixed Image
fixed_image = itk.GetArrayFromImage(fixed_image)
# Add that image as first index of registered image
registered_images.insert(0,fixed_image)

final_registered = np.stack(registered_images,axis=-1)
source_img = np.repeat(ref_img,final_registered.shape[-1],axis=-1).reshape(
ref_img.shape[0],
ref_img.shape[1],-1)

fix_images = sitk.GetImageFromArray(source_img)
move_images = sitk.GetImageFromArray(final_registered)


In [31]:
def display_images_with_alpha(image_z,alpha,fixed,moving):

    # print(moving[image_z,:,:])
    # print(fixed)

    img = (1.0-alpha) * fixed[image_z,:,:]+ alpha * moving[image_z,:,:]
    plt.imshow(sitk.GetArrayViewFromImage(img),cmap='inferno')
    plt.axis('off')
    plt.show()

In [32]:
%matplotlib tk
interact(display_images_with_alpha,
         image_z=(0,fix_images.GetSize()[0]-1),
         alpha = (0,1.0,0.05),
         fixed = fixed(fix_images),
         moving = fixed(move_images)
         )

interactive(children=(IntSlider(value=3978, description='image_z', max=7957), FloatSlider(value=0.5, descripti…

<function __main__.display_images_with_alpha(image_z, alpha, fixed, moving)>

In [ ]:
# You can obtain the transforation matrix between each frames
idx = 0
transform_matrix[idx].GetMatrix()

itkMatrixD22 ([[0.9814825174825168, -0.020060880296176142], [0.013426573426571773, 0.9728737145207712]])

In [ ]:

# Save the registered video
np.savez(os.path.join(ROOT,f"{video.split('.')[0]}_registered.npz"), cropped_vid=final_registered)